In [3]:
# Cell 1 - Setup
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/ML_final')

!pip install -q prophet

import pandas as pd
import numpy as np

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -q dagshub mlflow

import dagshub
import mlflow

from mlflow import MlflowClient
from mlflow.entities import ViewType


# Connect to the DagsHub MLflow server
dagshub.init(
    repo_owner="skupr23",
    repo_name="ml_final",
    mlflow=True,
)

EXPERIMENT_NAME = "Prophet_Training"


# Close a run left open by an interrupted notebook execution
if mlflow.active_run() is not None:
    mlflow.end_run()


client = MlflowClient()

# Include active and deleted experiments in the search
matches = client.search_experiments(
    view_type=ViewType.ALL,
    filter_string=f"name = '{EXPERIMENT_NAME}'",
)

if matches:
    experiment = matches[0]

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)
        print(
            f"Restored deleted experiment: "
            f"{EXPERIMENT_NAME} ({experiment.experiment_id})"
        )

# Creates the experiment only if it genuinely does not exist
mlflow.set_experiment(EXPERIMENT_NAME)

active_experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("Tracking URI :", mlflow.get_tracking_uri())
print("Experiment   :", active_experiment.name)
print("Experiment ID:", active_experiment.experiment_id)
print("Lifecycle    :", active_experiment.lifecycle_stage)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=05163060-2f21-4bd4-8b09-51c429828741&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7005906c0b032417d2bd6013597b421fc704b963ee95957231e367a7787a39f9




Accessing as skupr23

Initialized MLflow to track repo "skupr23/ml_final"

Repository skupr23/ml_final initialized!

Restored deleted experiment: Prophet_Training (8)
Tracking URI : https://dagshub.com/skupr23/ml_final.mlflow
Experiment   : Prophet_Training
Experiment ID: 8
Lifecycle    : active


In [5]:
# Cell 2 - Verify everything imports cleanly
from prophet import Prophet
print('Prophet imported OK')

Prophet imported OK


In [6]:
# Cell 3 - Load processed data
train = pd.read_pickle('data/train_processed.pkl')
train = train.sort_values(['Store','Dept','Date'])
print(train['Date'].min(), train['Date'].max())

2010-02-05 00:00:00 2012-10-26 00:00:00


In [7]:
# Cell 4 - WMAE metric
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [8]:
# Cell 5 - Prophet has a built-in mechanism for holiday effects - pass it
# the exact holiday dates already identified in feature engineering
# instead of relying on Prophet's generic US-holidays calendar, since
# Walmart's four flagged holidays are what the competition actually scores
# 5x weight on.
superbowl = ['2010-02-12','2011-02-11','2012-02-10','2013-02-08']
laborday  = ['2010-09-10','2011-09-09','2012-09-07','2013-09-06']
thanksgiving = ['2010-11-26','2011-11-25','2012-11-23','2013-11-29']
christmas = ['2010-12-31','2011-12-30','2012-12-28','2013-12-27']

holidays_df = pd.concat([
    pd.DataFrame({'holiday': 'SuperBowl', 'ds': pd.to_datetime(superbowl)}),
    pd.DataFrame({'holiday': 'LaborDay', 'ds': pd.to_datetime(laborday)}),
    pd.DataFrame({'holiday': 'Thanksgiving', 'ds': pd.to_datetime(thanksgiving)}),
    pd.DataFrame({'holiday': 'Christmas', 'ds': pd.to_datetime(christmas)}),
])
holidays_df['lower_window'] = -1
holidays_df['upper_window'] = 1
holidays_df.head()

,holiday,ds,lower_window,upper_window
0,SuperBowl,2010-02-12,-1,1
1,SuperBowl,2011-02-11,-1,1
2,SuperBowl,2012-02-10,-1,1
3,SuperBowl,2013-02-08,-1,1
0,LaborDay,2010-09-10,-1,1


---
# Cell 6 - Fit and score Prophet on the same population as XGBoost / LightGBM

Prophet is a **local** model: it learns one model per time series, with no
information shared across series. That makes a full sweep expensive, but scoring
a subsample is not an option - WMAE is an absolute-dollar metric, so a subsample
skewed toward low-volume departments scores low for reasons that have nothing to
do with forecast quality, and the result cannot be placed next to the GBM numbers.

So the holdout below is rebuilt *exactly* as the XGBoost and LightGBM notebooks
build it

    val = train[train['Date'] >= train['Date'].max() - 39 weeks]

which is every Store-Dept row in the window 2012-01-27 -> 2012-10-26 (40 weekly
dates). No minimum-length filter, no `asfreq` reindexing, no interpolated values:
every row scored is a real observation. One Prophet is fitted per Store-Dept pair
on that pair's own history and predicted on that pair's real validation dates.

Notes:

* **This fits ~3,300 Prophet models and takes roughly 40-80 minutes on Colab.**
  Set `MAX_SERIES` to a small number first, confirm the baseline check below
  passes, then set it back to `None` for the real run.
* Predictions come from `model.predict()` on the actual validation dates rather
  than `make_future_dataframe()`, which builds a regular date grid and would
  misalign against series with gapped weeks.
* Pairs with too little history for Prophet fall back to a median, computed
  **from the fit portion only** so nothing leaks from the holdout. (The GBM
  notebooks' own baseline uses `storedept_median_sales`, which is computed over
  all of train *including* the holdout - this fallback is the leak-free version.)

**The baseline is the population check.** It depends only on which rows are being
scored, not on Prophet, so it must reproduce the `1789.9133525504185` printed by
both GBM notebooks. If it does, the population is identical row for row and the
Prophet WMAE below is directly comparable to XGBoost's 1281.08 and LightGBM's
1297.44. If it does not, stop - nothing downstream means anything.

In [9]:
# Cell 6 - Rebuild the XGBoost/LightGBM holdout exactly, then fit and score Prophet on it
import time
import logging

from prophet.serialize import model_to_json

logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

VAL_WEEKS = 39         # the GBM notebooks' window; yields 40 weekly dates
MAX_SERIES = None      # cap the number of Store-Dept pairs while iterating; None = all
MIN_FIT_WEEKS = 2      # Prophet needs at least two points; below this we fall back
KEEP_MODELS = True     # retain fitted models (serialized to JSON, see Cell 7)
PROGRESS_EVERY = 250

PROPHET_KWARGS = dict(
    holidays=holidays_df,
    yearly_seasonality=True,
    weekly_seasonality=False,   # data is already weekly-aggregated
    daily_seasonality=False,
    seasonality_mode='additive',
)

# --- the split, exactly as the GBM notebooks compute it ---
_t = train.sort_values('Date')
VAL_START = _t['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
gbm_val = _t[_t['Date'] >= VAL_START].copy()
gbm_fit = _t[_t['Date'] < VAL_START].copy()

print('fit ends:  ', gbm_fit['Date'].max())
print('val range: ', gbm_val['Date'].min(), 'to', gbm_val['Date'].max())
print('val rows:  ', len(gbm_val),
      '| val dates:', gbm_val['Date'].nunique(),
      '| val pairs:', gbm_val[['Store', 'Dept']].drop_duplicates().shape[0])

# --- population check: this MUST reproduce the GBM notebooks' baseline ---
baseline_pred = gbm_val['lag_52'].fillna(gbm_val['storedept_median_sales'])
baseline_wmae = wmae(gbm_val['Weekly_Sales'], baseline_pred, gbm_val['IsHoliday'])
print('\n52-week seasonal baseline WMAE:', baseline_wmae)
print('   XGBoost / LightGBM reported:  1789.9133525504185')
if MAX_SERIES is None and abs(baseline_wmae - 1789.9133525504185) > 1e-6:
    raise AssertionError('population mismatch: this is not the GBM validation set')

# --- leak-free fallbacks, computed on the fit portion only ---
sd_median = gbm_fit.groupby(['Store', 'Dept'])['Weekly_Sales'].median().to_dict()
d_median = gbm_fit.groupby('Dept')['Weekly_Sales'].median().to_dict()
g_median = float(gbm_fit['Weekly_Sales'].median())

fit_groups = {k: g for k, g in gbm_fit.groupby(['Store', 'Dept'], sort=False)}

# --- one Prophet per pair, predicted on that pair's real validation dates ---
gbm_val = gbm_val.sort_values(['Store', 'Dept', 'Date'])
pair_groups = list(gbm_val.groupby(['Store', 'Dept'], sort=False))
if MAX_SERIES is not None:
    pair_groups = pair_groups[:MAX_SERIES]

prophet_models = {}    # (store, dept) -> JSON string, see Cell 7
pred_parts = []
n_prophet = n_fallback = n_failed = 0
start = time.time()

for i, ((store, dept), vg) in enumerate(pair_groups, 1):
    fg = fit_groups.get((store, dept))
    use_fallback = fg is None or len(fg) < MIN_FIT_WEEKS

    if not use_fallback:
        try:
            model = Prophet(**PROPHET_KWARGS)
            model.fit(fg[['Date', 'Weekly_Sales']].rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'}))
            # predict on the actual val dates, not make_future_dataframe: this is
            # gap-safe and guarantees the prediction lines up with gbm_val's rows
            yhat = model.predict(pd.DataFrame({'ds': vg['Date'].to_numpy()}))['yhat'].to_numpy()
            if KEEP_MODELS:
                prophet_models[(store, dept)] = model_to_json(model)
            n_prophet += 1
        except Exception:
            use_fallback = True
            n_failed += 1

    if use_fallback:
        v = sd_median.get((store, dept), np.nan)
        if not np.isfinite(v):
            v = d_median.get(dept, np.nan)
        if not np.isfinite(v):
            v = g_median
        yhat = np.full(len(vg), float(v))
        n_fallback += 1

    pred_parts.append(pd.Series(yhat, index=vg.index))

    if i % PROGRESS_EVERY == 0:
        print(f'  {i}/{len(pair_groups)} pairs | {time.time()-start:.0f}s elapsed')

elapsed = time.time() - start

all_pred_s = pd.concat(pred_parts)
scored = gbm_val.loc[all_pred_s.index]
all_pred = all_pred_s.to_numpy()
all_true = scored['Weekly_Sales'].to_numpy()
all_holiday = scored['IsHoliday'].to_numpy().astype(bool)

prophet_wmae = wmae(all_true, all_pred, all_holiday)
prophet_wmae_holiday = wmae(all_true[all_holiday], all_pred[all_holiday], all_holiday[all_holiday])
prophet_wmae_non_holiday = wmae(all_true[~all_holiday], all_pred[~all_holiday], all_holiday[~all_holiday])

# per-pair breakdown: which pairs Prophet handles well, and how that tracks volume.
# WMAE per pair = sum(w * |err|) / sum(w), vectorized rather than groupby.apply,
# which is slow across ~3,300 groups.
_s = scored.assign(pred=all_pred)
_s['_w'] = np.where(all_holiday, 5, 1)
_s['_wae'] = _s['_w'] * (_s['Weekly_Sales'] - _s['pred']).abs()
per_pair = (
    _s.groupby(['Store', 'Dept'])
      .agg(_wae_sum=('_wae', 'sum'),
           _w_sum=('_w', 'sum'),
           mean_weekly_sales=('Weekly_Sales', 'mean'),
           n_val_weeks=('Weekly_Sales', 'size'))
      .reset_index()
)
per_pair['wmae'] = per_pair['_wae_sum'] / per_pair['_w_sum']
per_pair = (per_pair[['Store', 'Dept', 'wmae', 'mean_weekly_sales', 'n_val_weeks']]
            .sort_values('wmae')
            .reset_index(drop=True))

population_complete = len(scored) == len(gbm_val)

print(f'\nFitted {n_prophet} Prophet models, {n_fallback} median fallbacks '
      f'({n_failed} of them after a fit error), in {elapsed:.0f}s')
print('Rows scored:', len(scored), '| full population:', population_complete)
print('\nBaseline (52wk):          ', baseline_wmae)
print('Prophet (full population):', prophet_wmae)
print('  XGBoost:                ', 1281.080964551936)
print('  LightGBM:               ', 1297.4414789901796)
print('\nHoliday WMAE:    ', prophet_wmae_holiday)
print('Non-holiday WMAE:', prophet_wmae_non_holiday)

fit ends:   2012-01-20 00:00:00
val range:  2012-01-27 00:00:00 to 2012-10-26 00:00:00
val rows:   118540 | val dates: 40 | val pairs: 3208

52-week seasonal baseline WMAE: 1789.9133525504185
   XGBoost / LightGBM reported:  1789.9133525504185
  250/3208 pairs | 73s elapsed
  500/3208 pairs | 151s elapsed
  750/3208 pairs | 199s elapsed
  1000/3208 pairs | 305s elapsed
  1250/3208 pairs | 352s elapsed
  1500/3208 pairs | 392s elapsed
  1750/3208 pairs | 452s elapsed
  2000/3208 pairs | 560s elapsed
  2250/3208 pairs | 645s elapsed
  2500/3208 pairs | 692s elapsed
  2750/3208 pairs | 781s elapsed
  3000/3208 pairs | 846s elapsed

Fitted 3167 Prophet models, 41 median fallbacks (0 of them after a fit error), in 947s
Rows scored: 118540 | full population: True

Baseline (52wk):           1789.9133525504185
Prophet (full population): 1905.4227245923344
  XGBoost:                 1281.080964551936
  LightGBM:                1297.4414789901796

Holiday WMAE:     1896.9936330616104
Non-holida

In [10]:
# Cell 7 - Save models locally (same pattern as LightGBM/XGBoost notebooks)
import os
import joblib

os.makedirs('models', exist_ok=True)

# ~3,300 fitted Prophet objects pickle to several GB, because each one carries its
# Stan backend. prophet.serialize.model_to_json drops the backend and keeps the
# parameters, which is what a forecast actually needs. Reload one with:
#     from prophet.serialize import model_from_json
#     model = model_from_json(prophet_models[(store, dept)])
joblib.dump(prophet_models, 'models/prophet_best.pkl')

size_mb = os.path.getsize('models/prophet_best.pkl') / 1e6
print(f'Saved {len(prophet_models)} models ({size_mb:.1f} MB).')

Saved 3167 models (79.9 MB).


In [11]:
# ============================================================
# MLflow - one experiment per architecture, one run per stage
# ============================================================
import os
import tempfile

PREPROCESSING_STEPS = [
    {'step': 'load_processed', 'detail': 'read data/train_processed.pkl, sort by Store-Dept-Date'},
    {'step': 'chronological_split',
     'detail': f"val = rows with Date >= max(Date) - {VAL_WEEKS} weeks; identical to the XGBoost/LightGBM notebooks"},
    {'step': 'no_series_filter', 'detail': 'every Store-Dept pair in the window is scored; no minimum-length filter'},
    {'step': 'no_resampling', 'detail': 'no asfreq / interpolation: every scored row is a real observation'},
    {'step': 'holiday_regressors',
     'detail': 'the four competition holidays passed to Prophet as an explicit holidays frame, +/-1 week window'},
    {'step': 'per_pair_fit', 'detail': "one Prophet per Store-Dept pair, fitted on that pair's history only"},
    {'step': 'fallback_aggregates',
     'detail': 'fit-portion median (store-dept -> dept -> global) for pairs with too little history'},
]


# ---------- run 1: cleaning / preprocessing ----------
with mlflow.start_run(run_name='Prophet_Cleaning'):
    mlflow.set_tag('stage', 'cleaning')
    mlflow.log_params({
        'val_weeks': VAL_WEEKS,
        'series_filter': 'none (all Store-Dept pairs in the validation window)',
        'resampling': 'none (no asfreq, no interpolation)',
        'merge_validate': 'many_to_one',
        'n_preprocessing_steps': len(PREPROCESSING_STEPS),
    })
    mlflow.log_metrics({
        'train_rows': train.shape[0],
        'train_cols': train.shape[1],
        'train_missing_cells': int(train.isna().sum().sum()),
        'n_stores': int(train['Store'].nunique()),
        'n_depts': int(train['Dept'].nunique()),
        'n_store_dept_pairs': int(train[['Store', 'Dept']].drop_duplicates().shape[0]),
        'fit_rows': len(gbm_fit),
        'val_rows': len(gbm_val),
        'val_dates': int(gbm_val['Date'].nunique()),
        'val_pairs': int(gbm_val[['Store', 'Dept']].drop_duplicates().shape[0]),
    })
    mlflow.log_dict({'steps': PREPROCESSING_STEPS}, 'preprocessing/pipeline.json')
    mlflow.log_dict({'columns': train.columns.tolist()}, 'preprocessing/processed_columns.json')
    mlflow.log_input(
        mlflow.data.from_pandas(train, name='train_processed', targets='Weekly_Sales'),
        context='training',
    )

# ---------- run 2: fitting (one Prophet model per Store-Dept pair) ----------
with mlflow.start_run(run_name='Prophet_Fitting'):
    mlflow.set_tag('stage', 'fitting')
    mlflow.log_params({
        'yearly_seasonality': True,
        'weekly_seasonality': False,   # data is already weekly-aggregated
        'daily_seasonality': False,
        'seasonality_mode': 'additive',
        'holidays': ['SuperBowl', 'LaborDay', 'Thanksgiving', 'Christmas'],
        'holiday_lower_window': -1,
        'holiday_upper_window': 1,
        'holiday_source': 'competition dates from feature_engineering.ipynb, not Prophet US calendar',
        'min_fit_weeks': MIN_FIT_WEEKS,
        'fallback': 'fit-portion median (store-dept -> dept -> global)',
    })
    mlflow.log_metrics({
        'n_holiday_rows': len(holidays_df),
        'n_prophet_models': n_prophet,
        'n_median_fallbacks': n_fallback,
        'n_fit_failures': n_failed,
        'fit_seconds': elapsed,
        'per_pair_wmae_mean': float(per_pair['wmae'].mean()),
        'per_pair_wmae_median': float(per_pair['wmae'].median()),
        'per_pair_wmae_min': float(per_pair['wmae'].min()),
        'per_pair_wmae_max': float(per_pair['wmae'].max()),
        # WMAE is absolute-dollar, so it tracks series volume. Logged so the
        # score can never be read as if it were scale-free.
        'per_pair_wmae_vs_volume_corr': float(per_pair['wmae'].corr(per_pair['mean_weekly_sales'])),
    })
    with tempfile.TemporaryDirectory() as tmp:
        p = os.path.join(tmp, 'per_pair_wmae.csv')
        per_pair.to_csv(p, index=False)
        mlflow.log_artifact(p, artifact_path='fitting')

# ---------- run 3: validation (chronological holdout) ----------
with mlflow.start_run(run_name='Prophet_Validation'):
    mlflow.set_tag('stage', 'validation')
    mlflow.set_tag('population', 'full' if population_complete else 'partial')
    mlflow.log_params({
        'scheme': 'chronological holdout (no k-fold: the target is a time series), rebuilt exactly as in the XGBoost/LightGBM notebooks',
        'val_weeks': VAL_WEEKS,
        'val_start': str(gbm_val['Date'].min().date()),
        'val_end': str(gbm_val['Date'].max().date()),
        'forecast_strategy': "one Prophet per Store-Dept pair, predicted on that pair's real validation dates",
        'max_series': str(MAX_SERIES),
        'population_complete': population_complete,
        'scored_on': 'same rows as XGBoost/LightGBM: directly comparable',
    })
    mlflow.log_metrics({
        'baseline_wmae': baseline_wmae,
        'final_wmae': prophet_wmae,
        'val_wmae_holiday': prophet_wmae_holiday,
        'val_wmae_non_holiday': prophet_wmae_non_holiday,
        'improvement_over_baseline': baseline_wmae - prophet_wmae,
        'val_rows': len(scored),
        'val_pairs': len(pair_groups),
    })

# ---------- run 4: final model ----------
with mlflow.start_run(run_name='Prophet_Final_Model') as final_run:
    mlflow.set_tags({'stage': 'final_model', 'algorithm': 'prophet'})
    mlflow.log_params({
        'model_type': 'local, one Prophet model per Store-Dept pair',
        'n_models': len(prophet_models),
        'seasonality_mode': 'additive',
        'val_weeks': VAL_WEEKS,
        'serialization': 'prophet.serialize.model_to_json, dict keyed by (Store, Dept)',
    })
    mlflow.log_metrics({
        'final_wmae': prophet_wmae,
        'baseline_wmae': baseline_wmae,
        'per_pair_wmae_median': float(per_pair['wmae'].median()),
    })

    # Not registered, and no mlflow.prophet flavor: this artifact is a dict of
    # per-pair Prophet models, not one estimator, so it is not interchangeable
    # with the WalmartSalesForecast pipeline versions.
    mlflow.log_artifact('models/prophet_best.pkl', artifact_path='estimator')
    print('Prophet final run:', final_run.info.run_id)

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run Prophet_Cleaning at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/4962f31ae53f44f39b9880818cc1ac04
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
🏃 View run Prophet_Fitting at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/30dc3dfc4dcc499bb79fc22bb4921d3f
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
🏃 View run Prophet_Validation at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/62d4266b42c44226b85b2666fe55ce23
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
Prophet final run: 28974c5d1f2c4995ba8ec11e557392cb
🏃 View run Prophet_Final_Model at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8/runs/28974c5d1f2c4995ba8ec11e557392cb
🧪 View experiment at: https://dagshub.com/skupr23/ml_final.mlflow/#/experiments/8
